# 06 - UniProt EGFR Target Metadata

Goal: collect target/protein metadata for EGFR so the Therapeutic Strategy Assistant can explain the biological target before listing therapies.

Flow: EGFR -> UniProt accession P00533 -> raw JSON -> cleaned target metadata CSV.

Outputs: `egfr_uniprot_target_metadata.csv`

### 1. Test notebook environment

In [23]:
import json
import sys
import time
from pathlib import Path

import pandas as pd
import requests

print("Notebook is working")
print("Python executable:", sys.executable)

Notebook is working
Python executable: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/.venv/bin/python


### 2. Set project folders

In [24]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "uniprot"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Raw data folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/uniprot
Processed data folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


### 3. Choose target

In [25]:
target_name = "EGFR"
uniprot_accession = "P00533"
uniprot_url = f"https://rest.uniprot.org/uniprotkb/{uniprot_accession}.json"

print("Target selected:", target_name)
print("UniProt accession:", uniprot_accession)

Target selected: EGFR
UniProt accession: P00533


### 4. Helper: GET JSON with retries

In [26]:
def get_json(url, params=None, retries=3, pause=2):
    """GET JSON with simple retries. Returns dict/list or None."""
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=(10, 60))
            if response.status_code == 200:
                return response.json()
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}, retrying...")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}, retrying...")
        time.sleep(pause)
    return None

### 5. Fetch UniProt target record and save raw JSON

In [27]:
uniprot_raw = get_json(uniprot_url)

raw_file = RAW_DIR / "egfr_uniprot_target_raw.json"
if uniprot_raw is None:
    raise RuntimeError("UniProt request failed; cannot build target metadata.")

with raw_file.open("w") as f:
    json.dump(uniprot_raw, f, indent=2)

print("Saved raw UniProt response:", raw_file)
print("Top-level keys:", list(uniprot_raw.keys())[:10])

Saved raw UniProt response: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/uniprot/egfr_uniprot_target_raw.json
Top-level keys: ['entryType', 'primaryAccession', 'secondaryAccessions', 'uniProtkbId', 'entryAudit', 'annotationScore', 'organism', 'proteinExistence', 'proteinDescription', 'genes']


### 6. Parse target metadata

In [28]:
def nested_get(value, path, default=None):
    current = value
    for key in path:
        if isinstance(current, dict):
            current = current.get(key)
        elif isinstance(current, list) and isinstance(key, int) and key < len(current):
            current = current[key]
        else:
            return default
    return default if current is None else current


def get_xrefs(record, database):
    return [xref.get("id") for xref in record.get("uniProtKBCrossReferences", []) if xref.get("database") == database]


function_texts = []
for comment in uniprot_raw.get("comments", []):
    if comment.get("commentType") == "FUNCTION":
        for text_obj in comment.get("texts", []):
            text = text_obj.get("value")
            if text:
                function_texts.append(text)

metadata_row = {
    "target_name": target_name,
    "uniprot_accession": uniprot_raw.get("primaryAccession"),
    "uniprot_id": uniprot_raw.get("uniProtkbId"),
    "protein_name": nested_get(uniprot_raw, ["proteinDescription", "recommendedName", "fullName", "value"]),
    "gene_name": nested_get(uniprot_raw, ["genes", 0, "geneName", "value"]),
    "organism": nested_get(uniprot_raw, ["organism", "scientificName"]),
    "function_summary": " ".join(function_texts),
    "ensembl_ids": " | ".join(get_xrefs(uniprot_raw, "Ensembl")),
    "chembl_ids": " | ".join(get_xrefs(uniprot_raw, "ChEMBL")),
    "hgnc_ids": " | ".join(get_xrefs(uniprot_raw, "HGNC")),
    "pdb_count": len(get_xrefs(uniprot_raw, "PDB")),
    "source": "UniProt",
    "url": f"https://www.uniprot.org/uniprotkb/{uniprot_accession}/entry",
}

uniprot_metadata_df = pd.DataFrame([metadata_row])
uniprot_metadata_df

,target_name,uniprot_accession,uniprot_id,protein_name,gene_name,organism,function_summary,ensembl_ids,chembl_ids,hgnc_ids,pdb_count,source,url
0,EGFR,P00533,EGFR_HUMAN,Epidermal growth factor receptor,EGFR,Homo sapiens,Receptor tyrosine kinase binding ligands of th...,ENST00000275493.7 | ENST00000342916.7 | ENST00...,CHEMBL203,HGNC:3236,354,UniProt,https://www.uniprot.org/uniprotkb/P00533/entry


### 7. Save processed target metadata

In [29]:
processed_file = PROCESSED_DIR / "egfr_uniprot_target_metadata.csv"
uniprot_metadata_df.to_csv(processed_file, index=False)

print("Saved:", processed_file)
print("Rows:", len(uniprot_metadata_df))

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_uniprot_target_metadata.csv
Rows: 1


### 8. Final result

In [30]:
print("UniProt Target Metadata Complete")
print("=" * 70)
print("Target:", target_name)
print("Rows:", len(uniprot_metadata_df))
print("Processed file:", processed_file)
display(uniprot_metadata_df)

UniProt Target Metadata Complete
Target: EGFR
Rows: 1
Processed file: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_uniprot_target_metadata.csv


,target_name,uniprot_accession,uniprot_id,protein_name,gene_name,organism,function_summary,ensembl_ids,chembl_ids,hgnc_ids,pdb_count,source,url
0,EGFR,P00533,EGFR_HUMAN,Epidermal growth factor receptor,EGFR,Homo sapiens,Receptor tyrosine kinase binding ligands of th...,ENST00000275493.7 | ENST00000342916.7 | ENST00...,CHEMBL203,HGNC:3236,354,UniProt,https://www.uniprot.org/uniprotkb/P00533/entry
